In [1]:
from __future__ import annotations
import csv
import re
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pydantic import BaseModel, Field
from typing import Dict, List, Optional, Literal, Any
import outlines

TOOLS_PATH = Path("data/ToolVerifier/tools.csv")
MODEL_NAME = "Qwen/Qwen3.5-27B"
DEVICE = "cuda:1"
DTYPE = "auto"

TOOL_INDEX = 0
REQUEST_COUNT = 8
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.8
TOP_P = 0.95

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class PropertySchema(BaseModel):
    type: str
    default: Optional[object] = None


class Parameters(BaseModel):
    type: Literal["object"] = "object"
    properties: Dict[str, PropertySchema] = Field(default_factory=dict)
    required: List[str] = Field(default_factory=list)


class ToolSchema(BaseModel):
    name: str
    description: str
    parameters: Parameters

In [3]:
SYSTEM_PROMPT = "You create short, realistic tool schemas."

def build_prompt(tool: Dict[str, Any]) -> str:
    name = tool.get("name", "").strip()
    description = tool.get("description", "").strip()

    return f"""Create a tool schema.

Tool name: {name}
Tool description: {description}

Return a realistic schema for this tool with:
- name (no spaces, use underscores)
- description
- parameters.type = "object"
- parameters.properties
- parameters.required (make sure to include all required parameters)

Keep it short and practical. DO NOT include any parameters that are not necessary for the tool to function.
"""

In [4]:
def load_tools(path: Path) -> List[Dict[str, Any]]:
    tools: List[Dict[str, Any]] = []

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)

        required = {"Name", "Description"}
        if reader.fieldnames is None or not required.issubset(set(reader.fieldnames)):
            raise ValueError(
                f"Expected CSV columns {required}, got {reader.fieldnames}"
            )

        for row in reader:
            tools.append({
                "name": row["Name"].strip(),
                "description": row["Description"].strip(),
            })

    return tools

def normalize_query(text: str) -> str:
    text = text.strip().strip('"').strip("'")
    text = re.sub(r"\s+", " ", text)
    return text

def resolve_dtype(dtype_name: str, device: str) -> torch.dtype:
    if dtype_name == "float16":
        return torch.float16
    if dtype_name == "bfloat16":
        return torch.bfloat16
    if dtype_name == "float32":
        return torch.float32
    if device.startswith("cpu"):
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_available() else torch.float32


def load_generator(model_name: str, device: str, dtype: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs: Dict[str, Any] = {
        "trust_remote_code": True,
        "torch_dtype": resolve_dtype(dtype, device),
    }
    if device == "auto":
        model_kwargs["device_map"] = "auto"

    hf_model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if device != "auto":
        hf_model = hf_model.to(device)
    hf_model.eval()

    structured_model = outlines.from_transformers(hf_model, tokenizer)
    return tokenizer, hf_model, structured_model

@torch.inference_mode()
def generate_queries(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    device: str,
) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = f"{SYSTEM_PROMPT}\n\n{prompt}"

    encoded = tokenizer(text, return_tensors="pt")
    if device != "auto":
        encoded = {key: value.to(device) for key, value in encoded.items()}

    generated = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_tokens = encoded["input_ids"].shape[1]
    new_tokens = generated[0][prompt_tokens:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def generate_schema(
    structured_model,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
) -> ToolSchema:
    full_prompt = f"{SYSTEM_PROMPT}\n\n{prompt}"
    print(f"Full prompt:\n{full_prompt}\n")

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        })
    else:
        generation_kwargs["do_sample"] = False

    raw_output = structured_model(full_prompt, ToolSchema, **generation_kwargs)
    print("Structured JSON:\n")
    print(raw_output)
    return ToolSchema.model_validate_json(raw_output)

In [5]:
tokenizer, hf_model, structured_model = load_generator(MODEL_NAME, DEVICE, DTYPE)

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 116803.40it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 851/851 [00:00<00:00, 1581.17it/s, Materializing param=model.norm.weight]                              


In [6]:
loaded_tools = load_tools(TOOLS_PATH)
query = build_prompt(loaded_tools[0])

In [7]:
# raw_output = generate_queries(
#     model=model,
#     tokenizer=tokenizer,
#     prompt=normalized_query,
#     max_new_tokens=MAX_NEW_TOKENS,
#     temperature=TEMPERATURE,
#     top_p=TOP_P,
#     device=DEVICE,
# )

schema = generate_schema(
    structured_model=structured_model,
    tokenizer=tokenizer,
    prompt=query,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
)
print("Parsed schema:\n")
print(schema.model_dump_json(indent=2))

Full prompt:
You create short, realistic tool schemas.

Create a tool schema.

Tool name: Bank Account Number generator
Tool description: The Bank Account Number tool generates a random bank account number for a bank.

Return a realistic schema for this tool with:
- name (no spaces, use underscores)
- description
- parameters.type = "object"
- parameters.properties
- parameters.required (make sure to include all required parameters)

Keep it short and practical. DO NOT include any parameters that are not necessary for the tool to function.


Structured JSON:

{ "name": "bank_account_number_generator", "description": "Generates a random bank account number for a bank.", "parameters": { "type": "object", "properties": { "bank_code": { "type": "string", "default": "123456" }, "account_length": { "type": "integer", "default": 10 } }, "required": [] } }
Parsed schema:

{
  "name": "bank_account_number_generator",
  "description": "Generates a random bank account number for a bank.",
  "para